<sub>**The Annotated ML Handbook** · Part III: Pandas · by Mahir Absar Khan</sub>

# Chapter 12: Cleaning data with pandas

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mahir-606/annotated-ml-handbook/blob/main/notebooks/part-3-pandas/12-pandas-cleaning.ipynb) &nbsp; [📚 Contents](../../book/README.md) &nbsp; [🌐 Web edition](https://mahir-606.github.io/annotated-ml-handbook/)

*Run the setup cell first, then read and run the cells from top to bottom. Every code cell is followed by a table explaining what each line does.*

In [1]:
# Repository setup (not part of the book's examples): run this cell first.
import os, sys
from pathlib import Path

# Files the examples create (CSV, models, figures) go into ./outputs next to the notebook.
OUT = Path.cwd() if Path.cwd().name == "outputs" else Path.cwd() / "outputs"
OUT.mkdir(exist_ok=True)
os.chdir(OUT)

# insurance.csv is the practice dataset created in Chapter 11 and reused by later chapters.
# If it is not here yet, copy it from the repository's data/ folder (or download it in Colab).
if not Path("insurance.csv").exists():
    import shutil, urllib.request
    for base in OUT.parents:
        if (base / "data" / "insurance.csv").exists():
            shutil.copy(base / "data" / "insurance.csv", "insurance.csv")
            break
    else:
        try:
            urllib.request.urlretrieve("https://raw.githubusercontent.com/Mahir-606/"
                                       "annotated-ml-handbook/main/data/insurance.csv", "insurance.csv")
        except Exception as e:
            print("Could not fetch insurance.csv; run Chapter 11 first.", e)

# The figure style used in the book.
import matplotlib.pyplot as plt
from cycler import cycler
plt.rcParams.update({
    "figure.figsize": (7.2, 4.0), "figure.dpi": 100,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.28, "grid.linewidth": 0.6,
    "axes.prop_cycle": cycler(color=["#1f4fff", "#e0245e", "#0f9b62", "#ff6a00",
                                     "#7b3ff2", "#0091b3", "#d99a00", "#d42a2a"]),
    "axes.titleweight": "bold", "axes.titlesize": 12.5, "axes.titlelocation": "left",
    "font.size": 10.5, "legend.frameon": False, "image.cmap": "viridis",
})

# Plotly: interactive in Jupyter/Colab; tools/run_all.py stores static PNGs so GitHub can show them.
try:
    import plotly.io as pio
    pio.renderers.default = os.environ.get("PLOTLY_RENDERER") or (
        "colab" if "google.colab" in sys.modules else "notebook_connected")
except ImportError:
    pass

Data scientists often say most of a project is spent cleaning data, and that is no exaggeration. Models are only as good as their input. This chapter is a toolbox for the problems real datasets have: missing values, duplicates, wrong types, messy text, inconsistent categories, dates stored as strings, and outliers.

## 12.1 A deliberately messy dataset · *Beginner*

> **📖 Definitions**
>
> **Data cleaning**: finding and fixing problems in raw data, such as wrong types, inconsistent labels, duplicates, missing and impossible values, before analysis.

**Read messy CSV text directly**

In [2]:
import io
import numpy as np
import pandas as pd

raw = """id,Name ,age,income,city,signup_date,score
1, Alice ,34,"52,000",new york,2025-01-15,88
2,bob,,61000,New York,2025/02/03,92
3,Carol,29,?,Boston,15-03-2025,
4,dave,41,48000,boston ,2025-04-22,75
4,dave,41,48000,boston ,2025-04-22,75
5,Eve,230,70000,Chicago,2025-05-30,81
6,frank,38,,CHICAGO,2025-06-11,-1
"""
df = pd.read_csv(io.StringIO(raw), na_values=["?"])
print(df)
print(df.dtypes)

   id    Name     age  income      city signup_date  score
0   1   Alice    34.0  52,000  new york  2025-01-15   88.0
1   2      bob    NaN   61000  New York  2025/02/03   92.0
2   3    Carol   29.0     NaN    Boston  15-03-2025    NaN
3   4     dave   41.0   48000   boston   2025-04-22   75.0
4   4     dave   41.0   48000   boston   2025-04-22   75.0
5   5      Eve  230.0   70000   Chicago  2025-05-30   81.0
6   6    frank   38.0     NaN   CHICAGO  2025-06-11   -1.0
id               int64
Name               str
age            float64
income             str
city               str
signup_date        str
score          float64
dtype: object


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>import io</code></td><td>Import the standard-library <code>io</code> module (in-memory files).</td></tr>
<tr><td><code>import numpy as np</code></td><td>Import NumPy (the array library) under the short name <code>np</code>.</td></tr>
<tr><td><code>import pandas as pd</code></td><td>Import pandas (the table library) under the short name <code>pd</code>.</td></tr>
<tr><td><code>&quot;&quot;&quot;</code></td><td>A small CSV with nearly every common problem.</td></tr>
<tr><td><code>df = pd.read_csv(io.StringIO(raw), na_values=[&quot;?&quot;])</code></td><td>io.StringIO makes a string behave like a file; &quot;?&quot; becomes NaN.</td></tr>
<tr><td><code>print(df)</code></td><td>Display the value of <code>df</code>.</td></tr>
<tr><td><code>print(df.dtypes)</code></td><td>Display the value of <code>df.dtypes</code>.</td></tr>
</table>
</details>

Spot the problems: a column name with a trailing space, names with odd capitalisation and spaces, a missing age, an income stored as text with a thousands separator, inconsistent city spellings, three date formats, a duplicated row, an impossible age (230), and a score of −1 that probably means "missing".

## 12.2 Step 1: tidy column names · *Beginner*

**Normalise headers**

In [3]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
print(df.columns.tolist())

['id', 'name', 'age', 'income', 'city', 'signup_date', 'score']


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>df.columns = df.columns.str.strip().str.lower().str.replace(&quot; &quot;, &quot;_&quot;)</code></td><td>Strip spaces, lowercase, snake_case.</td></tr>
<tr><td><code>print(df.columns.tolist())</code></td><td>[&#x27;id&#x27;, &#x27;name&#x27;, &#x27;age&#x27;, &#x27;income&#x27;, &#x27;city&#x27;, &#x27;signup_date&#x27;, &#x27;score&#x27;]</td></tr>
</table>
</details>

## 12.3 Step 2: duplicates · *Beginner*

**Find and drop duplicate rows**

In [4]:
print(df.duplicated().sum())
print(df[df.duplicated(keep=False)])
df = df.drop_duplicates().reset_index(drop=True)
print(df["id"].is_unique, len(df))

1
   id  name   age income     city signup_date  score
3   4  dave  41.0  48000  boston   2025-04-22   75.0
4   4  dave  41.0  48000  boston   2025-04-22   75.0
True 6


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>print(df.duplicated().sum())</code></td><td>Count rows that exactly repeat an earlier row.</td></tr>
<tr><td><code>print(df[df.duplicated(keep=False)])</code></td><td>keep=False marks ALL copies, so you can inspect them.</td></tr>
<tr><td><code>df = df.drop_duplicates().reset_index(drop=True)</code></td><td>Keep the first occurrence; renumber the index.</td></tr>
<tr><td><code>print(df[&quot;id&quot;].is_unique, len(df))</code></td><td>Check that the ID column is now unique.</td></tr>
</table>
</details>

> **🔬 Research practice** · **Check duplicates on IDs too**
>
> Exact-duplicate rows are easy. Harder: the same subject appearing twice with slightly different values, or the same patient in both your training and test sets. `df["id"].duplicated().sum()` catches the first; [Chapter 23](../part-5-scikit-learn/23-sklearn-model-selection.ipynb) handles the second with grouped splits.

## 12.4 Step 3: fix types · *Beginner*

**Text to numbers**

In [5]:
df["income"] = pd.to_numeric(df["income"].str.replace(",", ""), errors="coerce")
print(df["income"].tolist(), df["income"].dtype)
df["score"] = df["score"].replace(-1, np.nan)
print(df["score"].tolist())

[52000.0, 61000.0, nan, 48000.0, 70000.0, nan] float64
[88.0, 92.0, nan, 75.0, 81.0, nan]


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>df[&quot;income&quot;] = pd.to_numeric(df[&quot;income&quot;].str.replace(&quot;,&quot;, &quot;&quot;), errors=&quot;coerce&quot;)</code></td><td>Remove separators, then convert; errors=&quot;coerce&quot; turns unparseable values into NaN.</td></tr>
<tr><td><code>print(df[&quot;income&quot;].tolist(), df[&quot;income&quot;].dtype)</code></td><td>Now real numbers (float64), with NaN where values were missing.</td></tr>
<tr><td><code>df[&quot;score&quot;] = df[&quot;score&quot;].replace(-1, np.nan)</code></td><td>A sentinel value (-1 for &quot;missing&quot;) must become a real NaN.</td></tr>
<tr><td><code>print(df[&quot;score&quot;].tolist())</code></td><td>The -1 is now NaN.</td></tr>
</table>
</details>

**Mixed date formats: the tempting shortcut is wrong**

In [6]:
guess = pd.to_datetime(df["signup_date"], format="mixed", dayfirst=True)
print(guess.dt.strftime("%Y-%m-%d").tolist())

['2025-01-15', '2025-03-02', '2025-03-15', '2025-04-22', '2025-05-30', '2025-11-06']


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>guess = pd.to_datetime(df[&quot;signup_date&quot;], format=&quot;mixed&quot;, dayfirst=True)</code></td><td>Parse each value separately, treating ambiguous dates as day-first...</td></tr>
<tr><td><code>print(guess.dt.strftime(&quot;%Y-%m-%d&quot;).tolist())</code></td><td>...but look: &quot;2025/02/03&quot; became 2 March and &quot;2025-06-11&quot; became 6 November!</td></tr>
</table>
</details>

`dayfirst=True` was applied even to the year-first ISO dates. Silent errors like this are worse than crashes. The safe approach is to parse each **known** format explicitly and combine the results:

**Parse each known format explicitly**

In [7]:
d = df["signup_date"]
parsed = pd.to_datetime(d, format="%Y-%m-%d", errors="coerce")
parsed = parsed.fillna(pd.to_datetime(d, format="%Y/%m/%d", errors="coerce"))
parsed = parsed.fillna(pd.to_datetime(d, format="%d-%m-%Y", errors="coerce"))
print(parsed.isna().sum(), "unparsed")
df["signup_date"] = parsed
print(df["signup_date"].dt.strftime("%Y-%m-%d").tolist())
print(df["signup_date"].dt.month.tolist(), df["signup_date"].dt.day_name().tolist()[:3])

0 unparsed
['2025-01-15', '2025-02-03', '2025-03-15', '2025-04-22', '2025-05-30', '2025-06-11']
[1, 2, 3, 4, 5, 6] ['Wednesday', 'Monday', 'Saturday']


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>d = df[&quot;signup_date&quot;]</code></td><td>A short name for the raw date strings.</td></tr>
<tr><td><code>parsed = pd.to_datetime(d, format=&quot;%Y-%m-%d&quot;, errors=&quot;coerce&quot;)</code></td><td>ISO dates; anything else becomes NaT (missing time).</td></tr>
<tr><td><code>parsed = parsed.fillna(pd.to_datetime(d, format=&quot;%Y/%m/%d&quot;, errors=&quot;coerce&quot;))</code></td><td>Fill the gaps with a second format...</td></tr>
<tr><td><code>parsed = parsed.fillna(pd.to_datetime(d, format=&quot;%d-%m-%Y&quot;, errors=&quot;coerce&quot;))</code></td><td>...and a third.</td></tr>
<tr><td><code>print(parsed.isna().sum(), &quot;unparsed&quot;)</code></td><td>Always check that nothing was left behind.</td></tr>
<tr><td><code>df[&quot;signup_date&quot;] = parsed</code></td><td>Replace the text column with real datetimes.</td></tr>
<tr><td><code>print(df[&quot;signup_date&quot;].dt.strftime(&quot;%Y-%m-%d&quot;).tolist())</code></td><td>Every date is now correct and in one format.</td></tr>
<tr><td><code>print(df[&quot;signup_date&quot;].dt.month.tolist(), df[&quot;signup_date&quot;].dt.day_name().tolist()[:3])</code></td><td>The .dt accessor extracts parts of dates.</td></tr>
</table>
</details>

> **⚠️ Pitfall** · **Ambiguous dates**
>
> `03/04/2025` is 3 April in most of the world and 4 March in the US. Pass an explicit `format=` whenever you know it, count the values that failed to parse, and check a few parsed dates by eye.

## 12.5 Step 4: clean text and categories · *Beginner*

**The .str accessor**

In [8]:
df["name"] = df["name"].str.strip().str.title()
df["city"] = df["city"].str.strip().str.title()
print(df[["name", "city"]])
print(df["city"].value_counts())
print(df["name"].str.contains("a", case=False).tolist())
print(df["name"].str.len().tolist(), df["name"].str[0].tolist())

    name      city
0  Alice  New York
1    Bob  New York
2  Carol    Boston
3   Dave    Boston
4    Eve   Chicago
5  Frank   Chicago
city
New York    2
Boston      2
Chicago     2
Name: count, dtype: int64
[True, False, True, True, False, True]
[5, 3, 5, 4, 3, 5] ['A', 'B', 'C', 'D', 'E', 'F']


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>df[&quot;name&quot;] = df[&quot;name&quot;].str.strip().str.title()</code></td><td>Remove surrounding spaces; capitalise each word.</td></tr>
<tr><td><code>df[&quot;city&quot;] = df[&quot;city&quot;].str.strip().str.title()</code></td><td>&quot;boston &quot;, &quot;CHICAGO&quot; -&gt; &quot;Boston&quot;, &quot;Chicago&quot;.</td></tr>
<tr><td><code>print(df[[&quot;name&quot;, &quot;city&quot;]])</code></td><td>The tidied text columns.</td></tr>
<tr><td><code>print(df[&quot;city&quot;].value_counts())</code></td><td>Three cities instead of six spellings.</td></tr>
<tr><td><code>print(df[&quot;name&quot;].str.contains(&quot;a&quot;, case=False).tolist())</code></td><td>Pattern tests (regex by default) give boolean masks.</td></tr>
<tr><td><code>print(df[&quot;name&quot;].str.len().tolist(), df[&quot;name&quot;].str[0].tolist())</code></td><td>String length; first character.</td></tr>
</table>
</details>

Map inconsistent labels to a canonical set with a dictionary:

**Recoding values**

In [9]:
s = pd.Series(["M", "male", "F", "female", "f", "unknown"])
mapping = {"m": "male", "male": "male", "f": "female", "female": "female"}
print(s.str.lower().map(mapping).tolist())
print(s.replace({"M": "male", "F": "female"}).tolist())

['male', 'male', 'female', 'female', 'female', nan]
['male', 'male', 'female', 'female', 'f', 'unknown']


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>s = pd.Series([&quot;M&quot;, &quot;male&quot;, &quot;F&quot;, &quot;female&quot;, &quot;f&quot;, &quot;unknown&quot;])</code></td><td>Inconsistent codes for the same two categories.</td></tr>
<tr><td><code>mapping = {&quot;m&quot;: &quot;male&quot;, &quot;male&quot;: &quot;male&quot;, &quot;f&quot;: &quot;female&quot;, &quot;female&quot;: &quot;female&quot;}</code></td><td>Lowercase spelling -&gt; canonical label.</td></tr>
<tr><td><code>print(s.str.lower().map(mapping).tolist())</code></td><td><code>map</code> with a dict: anything not in the dict becomes NaN.</td></tr>
<tr><td><code>print(s.replace({&quot;M&quot;: &quot;male&quot;, &quot;F&quot;: &quot;female&quot;}).tolist())</code></td><td><code>replace</code> only changes listed values and leaves the rest alone.</td></tr>
</table>
</details>

## 12.6 Step 5: missing values · *Beginner*

> **📖 Definitions**
>
> **Missing value**: an absent observation, shown as `NaN`, `None` or `NaT` (for dates) in pandas.
>
> **Imputation**: replacing missing values with estimates, such as the mean, the median, the most frequent value, or a model's prediction.


First measure, then decide. There are three broad options: **drop**, **fill (impute)**, or **flag**.

**Measure missingness**

In [10]:
print(df.isna().sum())
print(df.isna().mean().round(2))
print(df[df.isna().any(axis=1)])

id             0
name           0
age            1
income         2
city           0
signup_date    0
score          2
dtype: int64
id             0.00
name           0.00
age            0.17
income         0.33
city           0.00
signup_date    0.00
score          0.33
dtype: float64
   id   name   age   income      city signup_date  score
1   2    Bob   NaN  61000.0  New York  2025-02-03   92.0
2   3  Carol  29.0      NaN    Boston  2025-03-15    NaN
5   6  Frank  38.0      NaN   Chicago  2025-06-11    NaN


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>print(df.isna().sum())</code></td><td>Missing count per column.</td></tr>
<tr><td><code>print(df.isna().mean().round(2))</code></td><td>Missing fraction per column.</td></tr>
<tr><td><code>print(df[df.isna().any(axis=1)])</code></td><td>Rows with at least one missing value.</td></tr>
</table>
</details>

**Drop or fill**

In [11]:
print(df.dropna().shape)
print(df.dropna(subset=["age"]).shape)
filled = df.copy()
filled["age"] = filled["age"].fillna(filled["age"].median())
filled["income"] = filled["income"].fillna(filled.groupby("city")["income"].transform("median"))
filled["score_missing"] = filled["score"].isna().astype(int)
filled["score"] = filled["score"].fillna(filled["score"].mean())
print(filled[["age", "income", "score", "score_missing"]])

(3, 7)
(5, 7)
     age   income  score  score_missing
0   34.0  52000.0   88.0              0
1   38.0  61000.0   92.0              0
2   29.0  48000.0   84.0              1
3   41.0  48000.0   75.0              0
4  230.0  70000.0   81.0              0
5   38.0  70000.0   84.0              1


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>print(df.dropna().shape)</code></td><td>Drop any row with a missing value: often far too aggressive.</td></tr>
<tr><td><code>print(df.dropna(subset=[&quot;age&quot;]).shape)</code></td><td>Drop only rows missing a critical column.</td></tr>
<tr><td><code>filled = df.copy()</code></td><td>Work on a copy so <code>df</code> keeps its missing values for comparison.</td></tr>
<tr><td><code>filled[&quot;age&quot;] = filled[&quot;age&quot;].fillna(filled[&quot;age&quot;].median())</code></td><td>Median imputation: robust to outliers.</td></tr>
<tr><td><code>filled[&quot;income&quot;] = filled[&quot;income&quot;].fillna(filled.groupby(&quot;city&quot;)[&quot;income&quot;].transform(&quot;median&quot;))</code></td><td>Fill with the median of the same city.</td></tr>
<tr><td><code>filled[&quot;score_missing&quot;] = filled[&quot;score&quot;].isna().astype(int)</code></td><td>Flag missingness: it can be informative in itself.</td></tr>
<tr><td><code>filled[&quot;score&quot;] = filled[&quot;score&quot;].fillna(filled[&quot;score&quot;].mean())</code></td><td>Mean imputation for score (after flagging which were missing).</td></tr>
<tr><td><code>print(filled[[&quot;age&quot;, &quot;income&quot;, &quot;score&quot;, &quot;score_missing&quot;]])</code></td><td>The filled columns and the missingness flag.</td></tr>
</table>
</details>

**Ordered data: forward fill and interpolation**

In [12]:
ts = pd.Series([1.0, np.nan, np.nan, 4.0, np.nan, 6.0])
print(ts.ffill().tolist())
print(ts.interpolate().tolist())

[1.0, 1.0, 1.0, 4.0, 4.0, 6.0]
[1.0, 2.0, 3.0, 4.0, 5.0, 6.0]


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>ts = pd.Series([1.0, np.nan, np.nan, 4.0, np.nan, 6.0])</code></td><td>An ordered series with gaps.</td></tr>
<tr><td><code>print(ts.ffill().tolist())</code></td><td>Carry the last known value forward (sensor readings, prices).</td></tr>
<tr><td><code>print(ts.interpolate().tolist())</code></td><td>Linear interpolation between known points.</td></tr>
</table>
</details>

> **🔬 Research practice** · **Impute inside the training data only**
>
> Computing a fill value (mean, median) on the full dataset before splitting leaks test-set information into training. In ML work, impute with scikit-learn's `SimpleImputer` inside a `Pipeline` ([Chapter 20](../part-5-scikit-learn/20-sklearn-preprocessing.ipynb)) so it is fitted on training folds only. Pandas filling is fine for exploration and for decisions that use no statistics, such as "missing means zero".

## 12.7 Step 6: outliers and impossible values · *Intermediate*

> **📖 Definitions**
>
> **Outlier**: an observation far from most others. It may be an error or a genuine but rare case.
>
> **IQR (interquartile range)**: Q3 − Q1, the spread of the middle 50% of the data. Values beyond 1.5 × IQR outside the quartiles are commonly flagged as outliers.

**Domain rules and the IQR rule**

In [13]:
print(df[(df["age"] < 0) | (df["age"] > 110)])
df.loc[df["age"] > 110, "age"] = np.nan
x = pd.Series([10, 12, 11, 13, 12, 95, 11, 10])
q1, q3 = x.quantile([0.25, 0.75])
iqr = q3 - q1
lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"bounds=({lo}, {hi}) outliers={x[(x < lo) | (x > hi)].tolist()}")
print(x.clip(lower=lo, upper=hi).tolist())

   id name    age   income     city signup_date  score
4   5  Eve  230.0  70000.0  Chicago  2025-05-30   81.0
bounds=(8.5, 14.5) outliers=[95]
[10.0, 12.0, 11.0, 13.0, 12.0, 14.5, 11.0, 10.0]


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>print(df[(df[&quot;age&quot;] &lt; 0) | (df[&quot;age&quot;] &gt; 110)])</code></td><td>Domain knowledge first: an age of 230 is a data-entry error.</td></tr>
<tr><td><code>df.loc[df[&quot;age&quot;] &gt; 110, &quot;age&quot;] = np.nan</code></td><td>Treat impossible values as missing rather than guessing.</td></tr>
<tr><td><code>x = pd.Series([10, 12, 11, 13, 12, 95, 11, 10])</code></td><td>One obvious outlier: 95.</td></tr>
<tr><td><code>q1, q3 = x.quantile([0.25, 0.75])</code></td><td>First and third quartiles.</td></tr>
<tr><td><code>iqr = q3 - q1</code></td><td>Interquartile range: the spread of the middle 50%.</td></tr>
<tr><td><code>lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr</code></td><td>Tukey&#x27;s fences: the rule behind boxplot whiskers.</td></tr>
<tr><td><code>print(f&quot;bounds=({lo}, {hi}) outliers={x[(x &lt; lo) | (x &gt; hi)].tolist()}&quot;)</code></td><td>The fences and the values outside them.</td></tr>
<tr><td><code>print(x.clip(lower=lo, upper=hi).tolist())</code></td><td><i>Winsorising</i>: cap extreme values instead of deleting them.</td></tr>
</table>
</details>

> **⚠️ Pitfall** · **Don't delete outliers by reflex**
>
> An outlier may be an error, or it may be the most interesting observation you have. Remove values that are *impossible* (an age of 230); investigate values that are merely *unusual*; and report what you did and why.

## 12.8 Transforming columns with map, apply and vectorized ops · *Intermediate*

**Fastest to slowest**

In [14]:
ins = pd.read_csv("insurance.csv")
ins["bmi_sq"] = ins["bmi"] ** 2
ins["obese"] = np.where(ins["bmi"] >= 30, "yes", "no")
ins["risk"] = np.select(
    [ins["smoker"].eq("yes") & (ins["bmi"] >= 30), ins["smoker"].eq("yes")],
    ["high", "medium"], default="low")
ins["region_code"] = ins["region"].map({"northeast": 0, "northwest": 1, "southeast": 2, "southwest": 3})
ins["desc"] = ins.apply(lambda r: f"{r['age']}y {r['sex']}", axis=1)
print(ins[["bmi", "obese", "smoker", "risk", "region_code", "desc"]].head())
print(ins["risk"].value_counts())

    bmi obese smoker risk  region_code        desc
0  27.9    no     no  low            1  58y female
1  39.4   yes     no  low            1    26y male
2  34.6   yes     no  low            3    19y male
3  17.3    no     no  low            2  48y female
4  23.8    no     no  low            3    35y male
risk
low       816
medium    103
high       81
Name: count, dtype: int64


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>ins = pd.read_csv(&quot;insurance.csv&quot;)</code></td><td>Load the practice dataset.</td></tr>
<tr><td><code>ins[&quot;bmi_sq&quot;] = ins[&quot;bmi&quot;] ** 2</code></td><td>1. Vectorized: always try this first.</td></tr>
<tr><td><code>ins[&quot;obese&quot;] = np.where(ins[&quot;bmi&quot;] &gt;= 30, &quot;yes&quot;, &quot;no&quot;)</code></td><td>2. np.where for two-way choices.</td></tr>
<tr><td><code>ins[&quot;risk&quot;] = np.select(</code></td><td>3. np.select for several conditions, checked in order.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;[ins[&quot;smoker&quot;].eq(&quot;yes&quot;) &amp; (ins[&quot;bmi&quot;] &gt;= 30), ins[&quot;smoker&quot;].eq(&quot;yes&quot;)],</code></td><td>The conditions, checked in order: smoker AND obese; then smoker.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;[&quot;high&quot;, &quot;medium&quot;], default=&quot;low&quot;)</code></td><td>The value for each condition; rows matching none get &quot;low&quot;.</td></tr>
<tr><td><code>ins[&quot;region_code&quot;] = ins[&quot;region&quot;].map({&quot;northeast&quot;: 0, &quot;northwest&quot;: 1, &quot;southeast&quot;: 2, &quot;southwest&quot;: 3})</code></td><td>4. map with a dict.</td></tr>
<tr><td><code>ins[&quot;desc&quot;] = ins.apply(lambda r: f&quot;{r[&#x27;age&#x27;]}y {r[&#x27;sex&#x27;]}&quot;, axis=1)</code></td><td>5. Row-wise apply: flexible but slow. Last resort.</td></tr>
<tr><td><code>print(ins[[&quot;bmi&quot;, &quot;obese&quot;, &quot;smoker&quot;, &quot;risk&quot;, &quot;region_code&quot;, &quot;desc&quot;]].head())</code></td><td>The new columns side by side.</td></tr>
<tr><td><code>print(ins[&quot;risk&quot;].value_counts())</code></td><td>How many customers fall into each risk group.</td></tr>
</table>
</details>

## 12.9 Categorical dtype · *Intermediate*

> **📖 Definitions**
>
> **Categorical variable**: a variable taking one of a limited set of labels, such as region or blood type. **Ordinal** categories have a natural order (S < M < L); **nominal** ones don't.

**Memory-efficient, ordered categories**

In [15]:
ins["region"] = ins["region"].astype("category")
print(ins["region"].cat.categories.tolist(), ins["region"].cat.codes[:5].tolist())
size = pd.Categorical(["M", "S", "L", "M"], categories=["S", "M", "L"], ordered=True)
print(size.sort_values().tolist(), (size > "S").tolist())
dummies = pd.get_dummies(ins["region"], prefix="reg", dtype=int)
print(dummies.head(3))

['northeast', 'northwest', 'southeast', 'southwest'] [1, 1, 3, 2, 3]
['S', 'M', 'M', 'L'] [True, False, True, True]
   reg_northeast  reg_northwest  reg_southeast  reg_southwest
0              0              1              0              0
1              0              1              0              0
2              0              0              0              1


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>ins[&quot;region&quot;] = ins[&quot;region&quot;].astype(&quot;category&quot;)</code></td><td>Stores each label once plus small integer codes.</td></tr>
<tr><td><code>print(ins[&quot;region&quot;].cat.categories.tolist(), ins[&quot;region&quot;].cat.codes[:5].tolist())</code></td><td>The category labels and the integer code stored for each row.</td></tr>
<tr><td><code>size = pd.Categorical([&quot;M&quot;, &quot;S&quot;, &quot;L&quot;, &quot;M&quot;], categories=[&quot;S&quot;, &quot;M&quot;, &quot;L&quot;], ordered=True)</code></td><td>An ORDERED category: S &lt; M &lt; L.</td></tr>
<tr><td><code>print(size.sort_values().tolist(), (size &gt; &quot;S&quot;).tolist())</code></td><td>Sorting and comparisons follow the declared order S &lt; M &lt; L.</td></tr>
<tr><td><code>dummies = pd.get_dummies(ins[&quot;region&quot;], prefix=&quot;reg&quot;, dtype=int)</code></td><td>One-hot encode a category: one 0/1 column per level.</td></tr>
<tr><td><code>print(dummies.head(3))</code></td><td>Four 0/1 columns, one per region.</td></tr>
</table>
</details>

## 12.10 A reusable cleaning function · *Intermediate*

Collect every step into a function, so cleaning is documented, repeatable and identical for new data.

**clean_customers()**

In [16]:
def parse_dates(s, formats=("%Y-%m-%d", "%Y/%m/%d", "%d-%m-%Y")):
    out = pd.Series(pd.NaT, index=s.index)
    for fmt in formats:
        out = out.fillna(pd.to_datetime(s, format=fmt, errors="coerce"))
    return out

def clean_customers(raw: pd.DataFrame) -> pd.DataFrame:
    """Return a cleaned copy of the raw customer table."""
    return (raw
            .rename(columns=lambda c: c.strip().lower().replace(" ", "_"))
            .drop_duplicates()
            .assign(
                name=lambda d: d["name"].str.strip().str.title(),
                city=lambda d: d["city"].str.strip().str.title(),
                income=lambda d: pd.to_numeric(d["income"].astype("str").str.replace(",", ""), errors="coerce"),
                age=lambda d: d["age"].where(d["age"].between(0, 110)),
                score=lambda d: d["score"].replace(-1, np.nan),
                signup_date=lambda d: parse_dates(d["signup_date"]),
            )
            .reset_index(drop=True))

clean = clean_customers(pd.read_csv(io.StringIO(raw), na_values=["?"]))
print(clean)
print(clean.dtypes)

   id   name   age   income      city signup_date  score
0   1  Alice  34.0  52000.0  New York  2025-01-15   88.0
1   2    Bob   NaN  61000.0  New York  2025-02-03   92.0
2   3  Carol  29.0      NaN    Boston  2025-03-15    NaN
3   4   Dave  41.0  48000.0    Boston  2025-04-22   75.0
4   5    Eve   NaN  70000.0   Chicago  2025-05-30   81.0
5   6  Frank  38.0      NaN   Chicago  2025-06-11    NaN
id                      int64
name                      str
age                   float64
income                float64
city                      str
signup_date    datetime64[ns]
score                 float64
dtype: object


<details open><summary><b>What each line does</b></summary>

<table>
<tr><td><code>def parse_dates(s, formats=(&quot;%Y-%m-%d&quot;, &quot;%Y/%m/%d&quot;, &quot;%d-%m-%Y&quot;)):</code></td><td>Define a function <code>parse_dates</code> that takes <code>s</code>, <code>formats</code>, <code>&quot;%Y/%m/%d&quot;</code> and <code>&quot;%d-%m-%Y&quot;)</code>.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;out = pd.Series(pd.NaT, index=s.index)</code></td><td>Start with all-missing dates.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;for fmt in formats:</code></td><td>Loop over <code>formats</code>, naming each item <code>fmt</code>.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;out = out.fillna(pd.to_datetime(s, format=fmt, errors=&quot;coerce&quot;))</code></td><td>Try each known format on the remaining gaps.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;return out</code></td><td>Send <code>out</code> back to the caller.</td></tr>
<tr><td><code>def clean_customers(raw: pd.DataFrame) -&gt; pd.DataFrame:</code></td><td>Takes a raw DataFrame and returns a cleaned one.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&quot;&quot;&quot;Return a cleaned copy of the raw customer table.&quot;&quot;&quot;</code></td><td>Docstring.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;return (raw</code></td><td>Send <code>(raw</code> back to the caller.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;.rename(columns=lambda c: c.strip().lower().replace(&quot; &quot;, &quot;_&quot;))</code></td><td>Tidy headers.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;.drop_duplicates()</code></td><td>Remove exact duplicates.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;.assign(</code></td><td>Each keyword creates or replaces a column.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;name=lambda d: d[&quot;name&quot;].str.strip().str.title(),</code></td><td>Tidy names.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;city=lambda d: d[&quot;city&quot;].str.strip().str.title(),</code></td><td>Tidy cities.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;income=lambda d: pd.to_numeric(d[&quot;income&quot;].astype(&quot;str&quot;).str.replace(&quot;,&quot;, &quot;&quot;), errors=&quot;coerce&quot;),</code></td><td>Text incomes -&gt; numbers (astype(&quot;str&quot;) first, in case the column was read as numbers).</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;age=lambda d: d[&quot;age&quot;].where(d[&quot;age&quot;].between(0, 110)),</code></td><td><code>where</code> keeps values meeting the condition, else NaN.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;score=lambda d: d[&quot;score&quot;].replace(-1, np.nan),</code></td><td>Sentinel -1 -&gt; NaN.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;signup_date=lambda d: parse_dates(d[&quot;signup_date&quot;]),</code></td><td>Parse the dates with the helper above.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;)</code></td><td>Close the call or tuple opened above.</td></tr>
<tr><td><code>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;.reset_index(drop=True))</code></td><td>Renumber rows 0..n-1 and close the chain.</td></tr>
<tr><td><code>clean = clean_customers(pd.read_csv(io.StringIO(raw), na_values=[&quot;?&quot;]))</code></td><td>Run the whole cleaning recipe in one call.</td></tr>
<tr><td><code>print(clean)</code></td><td>Display the value of <code>clean</code>.</td></tr>
<tr><td><code>print(clean.dtypes)</code></td><td>Display the value of <code>clean.dtypes</code>.</td></tr>
</table>
</details>

> **💡 Tip** · **Method chaining**
>
> Wrapping a chain in parentheses lets you put one operation per line. Using `lambda d:` inside `assign` refers to the DataFrame *at that point in the chain*, so later steps see the results of earlier ones.

> **✍️ Try it yourself**
>
> 1. In `insurance.csv`, fill missing BMI with the median BMI of the same sex and smoker group (`groupby(...).transform("median")`).
> 2. Count how many charges are outliers by the IQR rule. Are they mostly smokers?
> 3. Write a check function that raises an error if any age is outside 18–64 or any charge is negative.
>
> *Worked solutions:* [solutions-parts-1-4.ipynb](../../solutions/solutions-parts-1-4.ipynb), section *Chapter 12*.

---

⬅️ [Chapter 11: pandas foundations: Series, DataFrames and selection](11-pandas-basics.ipynb) · [📚 Contents](../../book/README.md) · [Chapter 13: Grouping, reshaping, joining and time series](13-pandas-transform.ipynb) ➡️